# Задача 1

Дан датафрейм, в котором существует колонка ’email’ (строка, длиной не более 255). Напишите функцию, которая в заданном датафрейме заменяет все невалидные email’ы на `"unknown@unknown.com"`. Валидным сичтается емэйл, выглядящий как `[name]@[subdomain].[subdomain]`, где `[name]` – строка, содержащая латинский буквы, точки и знаки - и _, `[subdomain]` и `[domain]` – строки, содержащие только латинские буквы и цифры. Длина строки `[domain]` – не больше 8 символов.

Хинт. Вопспользуйтесь методом `.apply()`, в который передайте функцию для обрабоки одного email’a.

При сдаче задания обязательно добавьте в конце вашей программы

```python
import sys
exec(sys.stdin.read())
```

## Решение

```python
import pandas as pd
import re
import sys


def fix_emails(df):
    """
    Функция получает датафрейм и заменяет все невалидные email
    в колонке 'email' на строку "unknown@unknown.com"
    """

    # --- ШАГ 1. Составляем «шаблон» (регулярное выражение) для правильного email ---
    #
    #   ^[a-zA-Z._-]+       →  [name] : латинские буквы, точка, дефис, подчёркивание
    #   @                   →  символ @
    #   [a-zA-Z0-9]+        →  [subdomain] : латинские буквы и цифры
    #   \.                  →  точка .
    #   [a-zA-Z0-9]{1,8}$   →  [domain] : латинские буквы и цифры, ДЛИНА от 1 до 8
    #
    pattern = r"^[a-zA-Z._-]+@[a-zA-Z0-9]+\.[a-zA-Z0-9]{1,8}$"

    # --- ШАГ 2. Вспомогательная функция: проверяет ОДИН email ---
    def check_one_email(email):
        # Сначала убеждаемся, что значение вообще является строкой
        # (иначе это может быть число, пустое значение NaN и т.д.)
        if isinstance(email, str) and re.match(pattern, email):
            return email  # email подошёл — оставляем как есть
        return "unknown@unknown.com"  # не подошёл — заменяем

    # --- ШАГ 3. Применяем нашу проверку к КАЖДОЙ строке колонки 'email' ---
    df["email"] = df["email"].apply(check_one_email)

    # Возвращаем изменённый датафрейм
    return df


# --- Эта строка нужна для автоматической проверки задания ---
exec(sys.stdin.read())
```

In [ ]:
import pandas as pd
import re


def fix_emails(df):
    # Шаблон для валидного email:
    # name: латинские буквы, точка, дефис, подчёркивание
    # @
    # subdomain: буквы и цифры
    # .
    # domain: буквы и цифры, длина от 1 до 8
    pattern = r'^[a-zA-Z._-]+@[a-zA-Z0-9]+\.[a-zA-Z0-9]{1,8}$'

    def check_one_email(email):
        if isinstance(email, str) and re.match(pattern, email):
            return email
        return 'unknown@unknown.com'

    df['email'] = df['email'].apply(check_one_email)
    return df


# ===== ТЕСТОВЫЙ ДАТАФРЕЙМ =====
test_emails = [
    'anna@mail.ru',           # валиден
    'john_doe@sub.com',       # валиден
    'mary.work@site.org',     # валиден
    'user-1@a.b',             # валиден
    'test@x.co',              # валиден
    'привет@mail.ru',         # невалиден: кириллица в name
    'test@@mail.ru',          # невалиден: два @
    'user@.com',              # невалиден: пустой subdomain
    'test@mail.commercial',   # невалиден: domain > 8 символов
    '123user@mail.ru',        # невалиден: цифры в name (не разрешены по условию)
    '@mail.ru',               # невалиден: пустое name
    'user@mail',              # невалиден: нет точки после subdomain
    'user name@mail.ru',      # невалиден: пробел в name
    '',                       # невалиден: пустая строка
    None                      # невалиден: не строка (NaN/None)
]

df_test = pd.DataFrame({'email': test_emails})

# ===== ЗАПУСК И ВЫВОД РЕЗУЛЬТАТА =====
print("ДО обработки:")
print(df_test)
print("\nПОСЛЕ обработки:")
df_fixed = fix_emails(df_test)
print(df_fixed)

ДО обработки:
                   email
0           anna@mail.ru
1       john_doe@sub.com
2     mary.work@site.org
3             user-1@a.b
4              test@x.co
5         привет@mail.ru
6          test@@mail.ru
7              user@.com
8   test@mail.commercial
9        123user@mail.ru
10              @mail.ru
11             user@mail
12     user name@mail.ru
13                      
14                   NaN

ПОСЛЕ обработки:
                  email
0          anna@mail.ru
1      john_doe@sub.com
2    mary.work@site.org
3   unknown@unknown.com
4             test@x.co
5   unknown@unknown.com
6   unknown@unknown.com
7   unknown@unknown.com
8   unknown@unknown.com
9   unknown@unknown.com
10  unknown@unknown.com
11  unknown@unknown.com
12  unknown@unknown.com
13  unknown@unknown.com
14  unknown@unknown.com


# Задача 2

Дан датафрейм, в котором существуют колонки ’age’ (возраст в годах, целое число) и ’income’ (дробное число). Напишите функцию, которая вернёт средний заработок людей младше 18 лет.

При сдаче задания обязательно добавьте в конце вашей программы

```python
import sys
exec(sys.stdin.read())
```

## Решение

```python
import pandas as pd
import sys


def average_income_under_18(df):
    """
    Функция принимает датафрейм и возвращает средний заработок
    людей, возраст которых меньше 18 лет.
    """

    # --- ШАГ 1. Отбираем только тех, кто младше 18 ---
    # df['age'] < 18  создаёт «маску» — столбец из True/False:
    #   True  — если возраст меньше 18
    #   False — если возраст 18 или больше
    #
    # df[ ... ] оставляет в датафрейме только строки с True
    young_people = df[df["age"] < 18]

    # --- ШАГ 2. Берём колонку 'income' у отобранных людей ---
    # и считаем среднее значение методом .mean()
    average = young_people["income"].mean()

    # Возвращаем результат
    return average


# --- Эта строка нужна для автоматической проверки задания ---
exec(sys.stdin.read())
```

In [ ]:
import pandas as pd


def average_income_under_18(df):
    young_people = df[df['age'] < 18]
    average = young_people['income'].mean()
    return average


# ===== ТЕСТОВЫЙ ДАТАФРЕЙМ =====
data = {
    'age':   [16, 25, 17, 42, 15, 18, 14, 30, 12, 19,
              17, 22, 16, 45, 13, 20, 11, 38, 17, 21],
    'income': [15000.0, 45000.0, 12000.0, 60000.0, 8000.0,
               20000.0, 5000.0, 55000.0, 3000.0, 25000.0,
               14000.0, 40000.0, 16000.0, 70000.0, 4500.0,
               28000.0, 2000.0, 50000.0, 13000.0, 32000.0]
}

df_test = pd.DataFrame(data)

# ===== ЗАПУСК И ВЫВОД РЕЗУЛЬТАТА =====
print("Весь датафрейм:")
print(df_test)
print("\nСредний заработок людей младше 18:")
result = average_income_under_18(df_test)
print(result)

Весь датафрейм:
    age   income
0    16  15000.0
1    25  45000.0
2    17  12000.0
3    42  60000.0
4    15   8000.0
5    18  20000.0
6    14   5000.0
7    30  55000.0
8    12   3000.0
9    19  25000.0
10   17  14000.0
11   22  40000.0
12   16  16000.0
13   45  70000.0
14   13   4500.0
15   20  28000.0
16   11   2000.0
17   38  50000.0
18   17  13000.0
19   21  32000.0

Средний заработок людей младше 18:
9250.0


# Задача 3

Дан датафрейм, в котором существуют колонки ’id’ (идентификатор пользователя, целое положительное число), ’age’ (возвраст в годах, целое число), ’sex’ (пол пользователя, строка, равная либо "Male", либо "Female", либо "Other"), "favorite_color" (любимый цвет пользователя, строка длиной не более 255). Найдите число различных любимых цветов у женщин младше 18 лет.

При сдаче задания обязательно добавьте в конце вашей программы

```python
import sys
exec(sys.stdin.read())
```

## Решение

```python
import pandas as pd
import sys


def count_unique_colors(df):
    """
    Функция считает, сколько различных любимых цветов
    у женщин младше 18 лет.
    """

    # --- ШАГ 1. Отбираем нужных людей ---
    # Нам нужны одновременно ДВА условия:
    #   1) пол женский  (sex == "Female")
    #   2) возраст < 18 (age < 18)
    #
    # В pandas каждое условие берётся в скобки,
    # а между ними ставится знак & (логическое И)
    young_women = df[(df["sex"] == "Female") & (df["age"] < 18)]

    # --- ШАГ 2. Берём колонку с цветами ---
    colors = young_women["favorite_color"]

    # --- ШАГ 3. Считаем количество различных (уникальных) значений ---
    # Метод .nunique() считает, сколько разных цветов встречается
    unique_count = colors.nunique()

    # Возвращаем число
    return unique_count


# --- Эта строка нужна для автоматической проверки задания ---
exec(sys.stdin.read())
```

In [ ]:
import pandas as pd


def count_unique_colors(df):
    young_women = df[(df['sex'] == 'Female') & (df['age'] < 18)]
    colors = young_women['favorite_color']
    unique_count = colors.nunique()
    return unique_count


# ===== ТЕСТОВЫЙ ДАТАФРЕЙМ =====
data = {
    'id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
           11, 12, 13, 14, 15],
    'age': [16, 25, 17, 42, 15, 18, 14, 30, 12, 19,
            17, 22, 16, 45, 13],
    'sex': ['Female', 'Male', 'Female', 'Female', 'Other',
            'Male', 'Female', 'Male', 'Female', 'Male',
            'Female', 'Other', 'Female', 'Male', 'Female'],
    'favorite_color': ['red', 'blue', 'blue', 'green', 'red',
                       'yellow', 'red', 'black', 'pink', 'blue',
                       'red', 'white', 'green', 'gray', 'pink']
}

df_test = pd.DataFrame(data)

# ===== ЗАПУСК И ВЫВОД РЕЗУЛЬТАТА =====
print("Весь датафрейм:")
print(df_test)

# Для наглядности: покажем только нужных людей и их цвета
young_women = df_test[(df_test['sex'] == 'Female') & (df_test['age'] < 18)]
print("\nЖенщины младше 18 и их цвета:")
print(young_women[['id', 'age', 'sex', 'favorite_color']])

print("\nЧисло различных любимых цветов у женщин младше 18:")
result = count_unique_colors(df_test)
print(result)